# 05_analysis - SQL Data Analysis (Airtel)

**Name:** Safran  |  **Register No:** 25181180078

Areas A-F. Run every cell top to bottom, then fill the **Result** and **Business interpretation** fields from YOUR output.

## Setup

In [66]:
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import display

# Same path logic as notebooks 00 and 02: find the project root that contains data/raw
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "raw").exists():
    for candidate in [PROJECT_ROOT.parent, PROJECT_ROOT / "notebook", PROJECT_ROOT / "notebooks"]:
        if (candidate / "data" / "raw").exists():
            PROJECT_ROOT = candidate
            break
DB_PATH = PROJECT_ROOT / "data" / "raw" / "airtel.db"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
print("Connected to database:", DB_PATH)

Connected to database: C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\data\raw\airtel.db


In [67]:
# Check the required tables/views exist (created in notebook 04_preparation)
objects = pd.read_sql_query("""
SELECT type, name FROM sqlite_master
WHERE type IN ('table','view') ORDER BY type, name;""", conn)
display(objects)
need = {"customer_features","recharges","usage_daily","v_usage_month","v_plans_clean"}
missing = need - set(objects["name"])
print("Missing objects:", missing if missing else "none")

,type,name
0,table,circles
1,table,complaints
2,table,customer_features
3,table,customers
4,table,plans
5,table,recharges
6,table,usage_daily
7,view,v_complaint_customer
8,view,v_plans_clean
9,view,v_revenue_customer


Missing objects: none


In [68]:
def run_sql(query):
    """Run a SQL query and display the result."""
    result = pd.read_sql_query(query, conn)
    display(result)
    return result

In [69]:
from pathlib import Path

sql_script = Path("C:\\Users\\thway\\OneDrive\\Desktop\\airtel-telecom-analytics\\sql\\practice\\06_customer_features.sql")

conn.executescript(sql_script.read_text(encoding="utf-8"))

print("customer_features table created successfully!")

customer_features table created successfully!


## AREA A - THE SUBSCRIBER BASE

In [70]:
# A1 - Subscribers by connection type
query_A1 = """
SELECT connection_type,                                   -- prepaid or postpaid
       COUNT(*) AS subs,                                  -- subscribers per group
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 2) AS pct          -- share of total base (dynamic denominator)
FROM customer_features                                    -- one row per subscriber
GROUP BY connection_type
ORDER BY subs DESC;
"""
result_A1 = run_sql(query_A1)

,connection_type,subs,pct
0,Prepaid,17149,77.95
1,Postpaid,4851,22.05


### A2 - Top 5 plans by subscribers

In [71]:
# A2 - Top 5 plans by subscribers
query_A2 = """
SELECT plan_name, plan_type, price_inr,                   -- plan details
       COUNT(*) AS subs                                   -- subscribers on plan
FROM customer_features
GROUP BY plan_name, plan_type, price_inr
ORDER BY subs DESC
LIMIT 5;
"""
result_A2 = run_sql(query_A2)

,plan_name,plan_type,price_inr,subs
0,Prepaid Annual 1799,Prepaid,1799,2901
1,Prepaid Quarterly 719,Prepaid,719,2561
2,Prepaid Max 479,Prepaid,479,2392
3,Prepaid Entertainment 399,Prepaid,399,2142
4,Prepaid Data 349,Prepaid,349,1884


### A3 - Top 5 circles by subscribers

In [72]:
# A3 - Top 5 circles by subscribers
query_A3 = """
SELECT circle_name, region, COUNT(*) AS subs              -- circle, region, subscribers
FROM customer_features
GROUP BY circle_name, region
ORDER BY subs DESC
LIMIT 5;
"""
result_A3 = run_sql(query_A3)

,circle_name,region,subs
0,Maharashtra,West,1428
1,Andhra Pradesh,South,1412
2,Mumbai,West,1395
3,Karnataka,South,1361
4,North East,East,1318


### A4 - Subscribers by age band

In [73]:
# A4 - Subscribers by age band
query_A4 = """
SELECT age_band, COUNT(*) AS subs,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 1) AS pct
FROM customer_features
GROUP BY age_band
ORDER BY age_band;
"""
result_A4 = run_sql(query_A4)

,age_band,subs,pct
0,18-24,5268,23.9
1,25-34,7467,33.9
2,35-44,4824,21.9
3,45-54,2895,13.2
4,55+,1546,7.0


### A5 - Subscribers by acquisition channel

In [74]:
# A5 - Subscribers by acquisition channel
query_A5 = """
SELECT acquisition_channel, COUNT(*) AS subs,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 1) AS pct
FROM customer_features
GROUP BY acquisition_channel
ORDER BY subs DESC;
"""
result_A5 = run_sql(query_A5)

,acquisition_channel,subs,pct
0,Retailer,9287,42.2
1,Airtel App,6513,29.6
2,Website,3081,14.0
3,Call Center,1770,8.0
4,Doorstep,1349,6.1


### A6 - Device type and OS combinations

In [75]:
# A6 - Device type and OS combinations
query_A6 = """
SELECT device_type, os, COUNT(*) AS subs
FROM customer_features
GROUP BY device_type, os
ORDER BY subs DESC;
"""
result_A6 = run_sql(query_A6)

,device_type,os,subs
0,Smartphone,Android,15613
1,Smartphone,iOS,3358
2,Feature Phone,Other,3029


### A7 - Tenure distribution

In [76]:
# A7 - Tenure distribution
query_A7 = """
SELECT tenure_years, COUNT(*) AS subs,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 1) AS pct
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
"""
result_A7 = run_sql(query_A7)

,tenure_years,subs,pct
0,0,4610,21.0
1,1,6961,31.6
2,2,7055,32.1
3,3,3374,15.3


### Analysis Summary - Area A

| Field | Your answer |
|---|---|
| Requirements completed | TODO |
| Headline finding | TODO |
| Decision supported | TODO |
| Biggest doubt / limitation | TODO |

## AREA B - REVENUE AND ARPU

In [77]:
# B1 - Average stated ARPU (limited measure)
query_B1 = """
SELECT ROUND(AVG(stated_arpu), 2) AS avg_arpu,            -- average plan-price ARPU
       COUNT(*) AS subs
FROM customer_features;
"""
result_B1 = run_sql(query_B1)

,avg_arpu,subs
0,421.65,22000


### B2 - Average actual ARPU and monthly total

In [78]:
# B2 - Average actual ARPU and monthly total
query_B2 = """
SELECT ROUND(AVG(actual_arpu_6m), 2) AS avg_actual,
       ROUND(SUM(actual_arpu_6m), 0) AS monthly_total
FROM customer_features;
"""
result_B2 = run_sql(query_B2)

,avg_actual,monthly_total
0,344.46,7578210.0


### B3 - Stated vs actual ARPU gap

In [79]:
# B3 - Stated vs actual ARPU gap
query_B3 = """
SELECT ROUND(AVG(stated_arpu), 2) AS stated,
       ROUND(AVG(actual_arpu_6m), 2) AS actual,
       ROUND(AVG(arpu_gap), 2) AS gap,
       ROUND(100.0 * (SUM(stated_arpu) - SUM(actual_arpu_6m))
             / NULLIF(SUM(actual_arpu_6m), 0), 2) AS over_pct   -- % over actual revenue
FROM customer_features;
"""
result_B3 = run_sql(query_B3)

,stated,actual,gap,over_pct
0,421.65,344.46,77.18,22.41


### B4 - Stated vs actual ARPU by connection type

In [80]:
# B4 - Stated vs actual ARPU by connection type
query_B4 = """
SELECT connection_type, COUNT(*) AS subs,
       ROUND(AVG(stated_arpu), 2) AS stated,
       ROUND(AVG(actual_arpu_6m), 2) AS actual
FROM customer_features
GROUP BY connection_type;
"""
result_B4 = run_sql(query_B4)

,connection_type,subs,stated,actual
0,Postpaid,4851,959.65,970.42
1,Prepaid,17149,269.46,167.40


### B5 - Top 5 plans by actual monthly revenue

In [81]:
# B5 - Top 5 plans by actual monthly revenue
query_B5 = """
SELECT plan_name, COUNT(*) AS subs,
       ROUND(SUM(actual_arpu_6m), 0) AS rev,
       ROUND(AVG(actual_arpu_6m), 2) AS per_sub
FROM customer_features
GROUP BY plan_name
ORDER BY rev DESC
LIMIT 5;
"""
result_B5 = run_sql(query_B5)

,plan_name,subs,rev,per_sub
0,Postpaid Infinity 1499,1431,2171756.0,1517.65
1,Postpaid Family 999,1239,1249246.0,1008.27
2,Postpaid 749,946,718959.0,760.00
3,Prepaid Entertainment 399,2142,671850.0,313.66
4,Prepaid Data 349,1884,520066.0,276.04


### B6 - Top 5 circles by actual monthly revenue

In [82]:
# B6 - Top 5 circles by actual monthly revenue
query_B6 = """
SELECT circle_name, COUNT(*) AS subs,
       ROUND(SUM(actual_arpu_6m), 0) AS rev
FROM customer_features
GROUP BY circle_name
ORDER BY rev DESC
LIMIT 5;
"""
result_B6 = run_sql(query_B6)

,circle_name,subs,rev
0,Mumbai,1395,473288.0
1,Karnataka,1361,468515.0
2,Andhra Pradesh,1412,463370.0
3,North East,1318,460998.0
4,Maharashtra,1428,450032.0


### B7 - Payments and amount, 1 Jan - 30 Jun 2025

> **Limits:** annual-plan renewals may be missing; June data may be incomplete.

In [83]:
# B7 - Payments and amount, 1 Jan - 30 Jun 2025
query_B7 = """
SELECT COUNT(*) AS payments,
       ROUND(SUM(amount_inr) / 10000000.0, 2) AS crore_inr   -- INR to crore
FROM recharges
WHERE recharge_date >= '2025-01-01'
  AND recharge_date <  '2025-07-01';                          -- includes whole of 30 June
"""
result_B7 = run_sql(query_B7)

,payments,crore_inr
0,83693,4.55


### B8 - Actual ARPU by tenure

In [84]:
# B8 - Actual ARPU by tenure
query_B8 = """
SELECT tenure_years, COUNT(*) AS subs,
       ROUND(AVG(actual_arpu_6m), 2) AS arpu
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
"""
result_B8 = run_sql(query_B8)

,tenure_years,subs,arpu
0,0,4610,329.10
1,1,6961,342.72
2,2,7055,350.13
3,3,3374,357.22


### B9 - Zero-recharge subscribers and churn

In [85]:
# B9 - Zero-recharge subscribers and churn
query_B9 = """
SELECT SUM(CASE WHEN recharge_count = 0 THEN 1 ELSE 0 END) AS paid_nothing,
       ROUND(100.0 * AVG(CASE WHEN recharge_count = 0 THEN is_churned END), 2) AS their_churn,
       ROUND(AVG(stated_arpu), 2) AS base_stated_arpu
FROM customer_features;
"""
result_B9 = run_sql(query_B9)

,paid_nothing,their_churn,base_stated_arpu
0,3711,44.3,421.65


### Analysis Summary - Area B

| Field | Your answer |
|---|---|
| Requirements completed | TODO |
| Headline finding | TODO |
| Decision supported | TODO |
| Biggest doubt / limitation | TODO |

## AREA C - RECHARGE BEHAVIOUR

### C1 - Payments and distinct subscribers by txn_type

In [86]:
# C1 - Payments and distinct subscribers by txn_type
query_C1 = """
SELECT txn_type, COUNT(*) AS payments,
       COUNT(DISTINCT customer_id) AS subs
FROM recharges
GROUP BY txn_type;
"""
result_C1 = run_sql(query_C1)

,txn_type,payments,subs
0,Bill Payment,108419,4851
1,Recharge,299670,17149


### C2 - Overall average payment (limited measure)

> **Care point:** one average mixes txn types. Compare with C3.

In [87]:
# C2 - Overall average payment (limited measure)
query_C2 = """
SELECT ROUND(AVG(amount_inr), 2) AS avg_payment
FROM recharges;
"""
result_C2 = run_sql(query_C2)

,avg_payment
0,532.59


### C3 - Average/min/max payment by txn_type

In [88]:
# C3 - Average/min/max payment by txn_type
query_C3 = """
SELECT txn_type,
       ROUND(AVG(amount_inr), 2) AS avg_amt,
       MIN(amount_inr) AS mn, MAX(amount_inr) AS mx
FROM recharges
GROUP BY txn_type;
"""
result_C3 = run_sql(query_C3)

,txn_type,avg_amt,mn,mx
0,Bill Payment,1082.42,399,1873
1,Recharge,333.66,99,1799


### C4 - Recharge frequency/total by connection type

In [89]:
# C4 - Recharge frequency/total by connection type
query_C4 = """
SELECT connection_type,
       ROUND(AVG(recharge_count), 2) AS payments,
       ROUND(AVG(recharge_total_6m), 0) AS total_6m
FROM customer_features
GROUP BY connection_type;
"""
result_C4 = run_sql(query_C4)

,connection_type,payments,total_6m
0,Postpaid,5.39,5823.0
1,Prepaid,3.36,1004.0


### C5 - Payment channels by transaction count

In [90]:
# C5 - Payment channels by transaction count
query_C5 = """
SELECT channel, COUNT(*) AS payments,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM recharges), 1) AS pct
FROM recharges
GROUP BY channel
ORDER BY payments DESC;
"""
result_C5 = run_sql(query_C5)

,channel,payments,pct
0,Website,111163,27.2
1,Airtel App,111083,27.2
2,USSD,75257,18.4
3,Retailer,74406,18.2
4,AutoPay,36180,8.9


### C6 - Channel share within each txn_type

In [91]:
# C6 - Channel share within each txn_type
query_C6 = """
SELECT txn_type, channel, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) /
             SUM(COUNT(*)) OVER (PARTITION BY txn_type), 1) AS pct   -- % within own txn_type
FROM recharges
GROUP BY txn_type, channel
ORDER BY txn_type, n DESC;
"""
result_C6 = run_sql(query_C6)

,txn_type,channel,n,pct
0,Bill Payment,Airtel App,36186,33.4
1,Bill Payment,AutoPay,36180,33.4
2,Bill Payment,Website,36053,33.3
3,Recharge,USSD,75257,25.1
4,Recharge,Website,75110,25.1
5,Recharge,Airtel App,74897,25.0
6,Recharge,Retailer,74406,24.8


### C7 - Churn by recharge_count (<= 8)

> **Limits:** comment on small groups at high/low counts.

In [92]:
# C7 - Churn by recharge_count (<= 8)
query_C7 = """
SELECT recharge_count, COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
WHERE recharge_count <= 8
GROUP BY recharge_count
ORDER BY recharge_count;
"""
result_C7 = run_sql(query_C7)

,recharge_count,subs,churn
0,0,3711,44.30
1,1,2762,40.04
2,2,2339,28.35
3,3,1026,54.09
4,4,714,76.05
5,5,3407,13.77
6,6,6732,1.52
7,7,507,5.52
8,8,32,90.63


## AREA D - DAILY USAGE OVER TIME

### D1 - Monthly traffic (TB) and active subscribers

In [93]:
pd.read_sql_query("""
SELECT name, type
FROM sqlite_master
WHERE name = 'v_usage_month';
""", conn)

,name,type
0,v_usage_month,view


In [94]:
# D1 - Monthly traffic (TB) and active subscribers
query_D1 = """
SELECT month,
       COUNT(DISTINCT customer_id) AS active_subs,
       ROUND(SUM(total_mb) / 1024.0 / 1024.0, 1) AS total_tb
FROM v_usage_month
GROUP BY month
ORDER BY month;
"""
result_D1 = run_sql(query_D1)

,month,active_subs,total_tb
0,2025-01,21302,338.0
1,2025-02,21162,307.3
2,2025-03,20366,332.5
3,2025-04,19320,310.3
4,2025-05,18356,363.1
5,2025-06,17326,338.9


### D2 - Average daily data across usage_daily rows (limited measure)

> **Care point:** averages subscriber-day rows (denominator = row count), not subscribers or calendar days. Compare with D3.

In [95]:
# D2 - Average daily data across usage_daily rows (limited measure)
query_D2 = """
SELECT ROUND(AVG(data_mb), 2) AS avg_mb,
       COUNT(*) AS rows_
FROM usage_daily;
"""
result_D2 = run_sql(query_D2)

,avg_mb,rows_
0,1572.87,1326739


### D3 - Per active day vs per observed day

In [96]:
# D3 - Per active day vs per observed day
query_D3 = """
SELECT ROUND(AVG(mb_per_active_day), 2) AS per_active_day,
       ROUND(AVG(mb_per_observed_day), 2) AS per_observed_day
FROM customer_features;
"""
result_D3 = run_sql(query_D3)

,per_active_day,per_observed_day
0,1404.44,524.05


### D4 - Average daily data by weekday

In [97]:
# D4 - Average daily data by weekday
query_D4 = """
SELECT strftime('%w', usage_date) AS dow,                 -- 0 = Sunday ... 6 = Saturday
       COUNT(*) AS rows_,
       ROUND(AVG(data_mb), 1) AS avg_mb
FROM usage_daily
GROUP BY dow
ORDER BY dow;
"""
result_D4 = run_sql(query_D4)

,dow,rows_,avg_mb
0,0,190368,1578.1
1,1,189914,1572.9
2,2,183823,1569.5
3,3,191088,1564.5
4,4,190764,1580.0
5,5,190542,1569.5
6,6,190240,1575.5


### D5 - Active subscribers by month

In [98]:
# D5 - Active subscribers by month
query_D5 = """
SELECT month, COUNT(DISTINCT customer_id) AS active_subs
FROM v_usage_month
GROUP BY month
ORDER BY month;
"""
result_D5 = run_sql(query_D5)

,month,active_subs
0,2025-01,21302
1,2025-02,21162
2,2025-03,20366
3,2025-04,19320
4,2025-05,18356
5,2025-06,17326


### D6 - Average voice and SMS by month

In [99]:
# D6 - Average voice and SMS by month
query_D6 = """
SELECT strftime('%Y-%m', usage_date) AS m,
       ROUND(AVG(voice_minutes), 1) AS avg_voice,
       ROUND(AVG(sms_count), 3) AS avg_sms
FROM usage_daily
GROUP BY m
ORDER BY m;
"""
result_D6 = run_sql(query_D6)

,m,avg_voice,avg_sms
0,2025-01,55.6,0.745
1,2025-02,55.8,0.746
2,2025-03,56.1,0.750
3,2025-04,56.5,0.760
4,2025-05,56.7,0.764
5,2025-06,57.1,0.772


### D7 - Monthly GB per subscriber and active days

In [100]:
# D7 - Monthly GB per subscriber and active days
query_D7 = """
SELECT month,
       ROUND(AVG(total_mb) / 1024.0, 2) AS avg_gb_per_sub,
       ROUND(AVG(usage_days), 1) AS avg_days
FROM v_usage_month
GROUP BY month
ORDER BY month;
"""
result_D7 = run_sql(query_D7)

,month,avg_gb_per_sub,avg_days
0,2025-01,16.25,11.4
1,2025-02,14.87,10.3
2,2025-03,16.72,11.5
3,2025-04,16.44,11.2
4,2025-05,20.26,11.7
5,2025-06,20.03,11.5


### D8 - Data per active day, never-churned subscribers

In [101]:
# D8 - Data per active day, never-churned subscribers
query_D8 = """
SELECT m.month,
       ROUND(AVG(m.total_mb * 1.0 / NULLIF(m.usage_days, 0)), 1) AS mb_per_day
FROM v_usage_month AS m
JOIN customer_features AS c ON c.customer_id = m.customer_id
WHERE c.is_churned = 0
GROUP BY m.month
ORDER BY m.month;
"""
result_D8 = run_sql(query_D8)

,month,mb_per_day
0,2025-01,1410.6
1,2025-02,1422.1
2,2025-03,1418.1
3,2025-04,1411.8
4,2025-05,1659.5
5,2025-06,1662.2


## AREA E - USAGE INTENSITY AND DATA ALLOWANCE

### E1 - Quartiles of daily data use

In [102]:
# E1 - Quartiles of daily data use
query_E1 = """
WITH n AS (
  SELECT mb_per_active_day AS m,
         NTILE(4) OVER (ORDER BY mb_per_active_day) AS q   -- 4 equal groups
  FROM customer_features
  WHERE usage_days > 0                                      -- exclude no-usage subs
)
SELECT q, COUNT(*) AS subs, ROUND(MIN(m), 0) AS mn, ROUND(MAX(m), 0) AS mx
FROM n
GROUP BY q
ORDER BY q;
"""
result_E1 = run_sql(query_E1)

,q,subs,mn,mx
0,1,5499,25.0,657.0
1,2,5499,657.0,1093.0
2,3,5499,1093.0,1572.0
3,4,5499,1572.0,9720.0


### E2 - Churn by daily-data quartile

> **Limits:** usage and churn measured in the same period (circular).

In [103]:
# E2 - Churn by daily-data quartile
query_E2 = """
WITH n AS (
  SELECT is_churned,
         NTILE(4) OVER (ORDER BY mb_per_active_day) AS q
  FROM customer_features
  WHERE usage_days > 0
)
SELECT q, COUNT(*) AS subs, ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM n
GROUP BY q
ORDER BY q;
"""
result_E2 = run_sql(query_E2)

,q,subs,churn
0,1,5499,30.28
1,2,5499,28.11
2,3,5499,22.51
3,4,5499,14.09


### E3 - Voice and data per active day by connection/device

In [104]:
# E3 - Voice and data per active day by connection/device
query_E3 = """
SELECT connection_type, device_type, COUNT(*) AS subs,
       ROUND(AVG(total_voice_min * 1.0 / NULLIF(usage_days, 0)), 1) AS voice_per_day,
       ROUND(AVG(mb_per_active_day), 0) AS mb_per_day
FROM customer_features
WHERE usage_days > 0
GROUP BY connection_type, device_type
ORDER BY voice_per_day DESC;
"""
result_E3 = run_sql(query_E3)

,connection_type,device_type,subs,voice_per_day,mb_per_day
0,Postpaid,Feature Phone,690,48.6,531.0
1,Postpaid,Smartphone,4161,48.0,3353.0
2,Prepaid,Feature Phone,2338,47.9,166.0
3,Prepaid,Smartphone,14807,47.9,1093.0


### E4 - Plan allowance vs actual monthly use

In [105]:
# E4 - Plan allowance vs actual monthly use
query_E4 = """
SELECT plan_name, monthly_gb,
       ROUND(AVG(total_data_mb) / 1024.0 / 6, 2) AS actual_gb_per_month
FROM customer_features
WHERE usage_days > 0
GROUP BY plan_name, monthly_gb
ORDER BY monthly_gb;
"""
result_E4 = run_sql(query_E4)

,plan_name,monthly_gb,actual_gb_per_month
0,Prepaid Lite 99,15.0,2.88
1,Prepaid Smart 155,30.0,5.72
2,Prepaid Value 209,30.0,5.89
3,Postpaid 399,40.0,7.86
4,Prepaid Max 479,45.0,8.84
5,Prepaid Popular 299,45.0,8.93
6,Prepaid Annual 1799,60.0,11.87
7,Prepaid Quarterly 719,60.0,11.78
8,Postpaid 499,75.0,15.47
9,Prepaid Data 349,75.0,15.13


### E5 - Allowance used and subscribers under half

In [106]:
# E5 - Allowance used and subscribers under half
query_E5 = """
SELECT ROUND(AVG(monthly_gb), 1) AS allowance_gb,
       ROUND(AVG(total_data_mb / 1024.0 / 6), 2) AS used_gb,
       ROUND(100.0 * AVG((total_data_mb / 1024.0 / 6) / NULLIF(monthly_gb, 0)), 1) AS pct_of_allowance,
       SUM(CASE WHEN (total_data_mb / 1024.0 / 6) < monthly_gb * 0.5 THEN 1 ELSE 0 END) AS under_half
FROM customer_features
WHERE usage_days > 0;
"""
result_E5 = run_sql(query_E5)

,allowance_gb,used_gb,pct_of_allowance,under_half
0,76.8,15.44,19.8,20560


### E6 - Spend, data and churn by m_score

In [107]:
# E6 - Spend, data and churn by monetary score (m_score built here with NTILE)
query_E6 = """
WITH scored AS (
    SELECT recharge_total_6m,
           mb_per_active_day,
           is_churned,
           -- 4 equal groups by six-month spend: 1 = lowest spend, 4 = highest
           NTILE(4) OVER (ORDER BY COALESCE(recharge_total_6m, 0)) AS m_score
    FROM customer_features
)
SELECT m_score,
       COUNT(*) AS subs,
       ROUND(AVG(recharge_total_6m), 0) AS spend_6m,
       ROUND(AVG(mb_per_active_day), 0) AS mb_per_day,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM scored
GROUP BY m_score
ORDER BY m_score;
"""
result_E6 = run_sql(query_E6)

,m_score,subs,spend_6m,mb_per_day,churn
0,1,5500,167.0,973.0,54.82
1,2,5500,945.0,778.0,19.96
2,3,5500,1678.0,1117.0,11.20
3,4,5500,5477.0,2750.0,9.04


### E7 - Churn by usage_days quartile

> **Limits:** churners cannot accumulate later usage days.

In [108]:
# E7 - Churn by usage_days quartile
query_E7 = """
WITH n AS (
  SELECT usage_days, is_churned,
         NTILE(4) OVER (ORDER BY usage_days) AS q
  FROM customer_features
)
SELECT q, COUNT(*) AS subs, MIN(usage_days) AS mn, MAX(usage_days) AS mx,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM n
GROUP BY q
ORDER BY q;
"""
result_E7 = run_sql(query_E7)

,q,subs,mn,mx,churn
0,1,5500,0,31,58.56
1,2,5500,31,57,24.45
2,3,5500,57,86,9.49
3,4,5500,86,180,2.51


### E8 - SMS per active day and never-SMS subscribers

In [109]:
# E8 - SMS per active day and never-SMS subscribers
query_E8 = """
SELECT ROUND(AVG(total_sms * 1.0 / NULLIF(usage_days, 0)), 3) AS sms_per_active_day,
       SUM(CASE WHEN total_sms = 0 THEN 1 ELSE 0 END) AS never_sent_one,
       COUNT(*) AS subs
FROM customer_features
WHERE usage_days > 0;
"""
result_E8 = run_sql(query_E8)

,sms_per_active_day,never_sent_one,subs
0,0.598,654,21996


## AREA F - ROAMING

### F1 - Roaming vs non-roaming days and subscribers

In [110]:
# F1 - Roaming vs non-roaming days and subscribers
query_F1 = """
SELECT is_roaming, COUNT(*) AS days,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM usage_daily), 2) AS pct,
       COUNT(DISTINCT customer_id) AS subs
FROM usage_daily
GROUP BY is_roaming;
"""
result_F1 = run_sql(query_F1)

,is_roaming,days,pct,subs
0,0,1287143,97.02,21994
1,1,39596,2.98,16233


### F2 - Data and voice on roaming vs home days

In [111]:
# F2 - Data and voice on roaming vs home days
query_F2 = """
SELECT is_roaming,
       ROUND(AVG(data_mb), 2) AS avg_mb,
       ROUND(AVG(voice_minutes), 2) AS avg_voice
FROM usage_daily
GROUP BY is_roaming;
"""
result_F2 = run_sql(query_F2)

,is_roaming,avg_mb,avg_voice
0,0,1572.95,56.27
1,1,1570.05,56.09


### F3 - Roaming by connection type

In [112]:
# F3 - Roaming by connection type
query_F3 = """
SELECT connection_type, COUNT(*) AS subs,
       ROUND(AVG(roaming_days), 2) AS avg_roam_days,
       ROUND(100.0 * AVG(CASE WHEN roaming_days > 0 THEN 1.0 ELSE 0 END), 1) AS pct_ever
FROM customer_features
WHERE usage_days > 0
GROUP BY connection_type;
"""
result_F3 = run_sql(query_F3)

,connection_type,subs,avg_roam_days,pct_ever
0,Postpaid,4851,1.84,74.8
1,Prepaid,17145,1.79,73.5


### F4 - Churn by roaming-day band (interpret with caution)

> **Care point:** more roaming days may just mean more active days. Compare with E7; no causal claim.

In [113]:
# F4 - Churn by roaming-day band (interpret with caution)
query_F4 = """
SELECT CASE WHEN roaming_days = 0 THEN '0'
            WHEN roaming_days <= 2 THEN '1-2'
            WHEN roaming_days <= 5 THEN '3-5'
            ELSE '6+' END AS band,
       COUNT(*) AS subs,
       ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features
WHERE usage_days > 0
GROUP BY band
ORDER BY band;
"""
result_F4 = run_sql(query_F4)

,band,subs,churn
0,0,5763,44.61
1,1-2,9922,21.58
2,3-5,5546,8.69
3,6+,765,3.92


### F5 - Top 5 circles by roaming share

In [114]:
# F5 - Top 5 circles by roaming share
query_F5 = """
SELECT circle_name,
       ROUND(100.0 * SUM(roaming_days) / NULLIF(SUM(usage_days), 0), 2) AS pct_roaming
FROM customer_features
WHERE usage_days > 0
GROUP BY circle_name
ORDER BY pct_roaming DESC
LIMIT 5;
"""
result_F5 = run_sql(query_F5)

,circle_name,pct_roaming
0,Odisha,3.07
1,Haryana,3.07
2,Delhi,3.07
3,Karnataka,3.05
4,Assam,3.05


In [115]:
conn.close()
print("Database connection closed.")

Database connection closed.
